# 基于并行广度优先遍历的网络故障诊断——实验实操

本实践围绕网络故障诊断，补全邻接表的数据结构定义、头插法构图、CSR 转换与 Ascend C 并行扩展中的关键代码。请先阅读任务说明和辅助代码，再根据注释完成 `TODO`。

完成后应能够：

1. 使用标准 C++ 定义顶点、边结点和图的邻接表结构；
2. 基于头插法建立无向图的邻接关系；
3. 沿指针遍历链表，将邻接关系转换为 CSR 数组；
4. 使用 Ascend C 并行扩展同一层顶点，由 Host 合并结果并核对 CPU 串行参考输出。


## 0. 实践介绍

### 0.1 实践任务

本实践共包含 4 个任务：

- **任务一：数据结构的定义——使用标准 C++ 定义邻接表**
- **任务二：图的构建——基于头插法构建图的邻接表**
- **任务三：图的 CSR 压缩存储——基于邻接表对图进行 CSR 压缩存储**
- **任务四：基于 Ascend C 实现并行 BFS——体会在昇腾 NPU 上实现并行 BFS 算法**

操作方式：

1. 先阅读任务说明；
2. 在带有 `TODO` 的代码单元中补全代码；
3. **重新运行该代码单元**，将修改后的 C++/Ascend C 代码写入文件；
4. 再运行对应的测试单元；
5. 测试通过后再进入下一项任务。

注：测试代码和工程辅助代码已经提供，不需要修改。代码单元需要从上至下依次运行。每处 TODO 只涉及一至两行核心代码；保留 TODO 注释不会影响测试通过。任务一至三只需要 `g++` 环境，任务四需要配置了 CANN 的昇腾 NPU 环境。任务四中的 CPU 串行 BFS 已完整提供，用作正确性和性能比较的参考。


In [ ]:
from pathlib import Path
import subprocess

PRACTICE_DIR = Path("Source/02.06").resolve()
for rel in ["cpp", "ascend_ops/op_kernel", "ascend_ops/host_launch", "scripts", "results"]:
    (PRACTICE_DIR / rel).mkdir(parents=True, exist_ok=True)

def test_cpp(task, sources):
    build_dir = PRACTICE_DIR / "build_cpp"
    build_dir.mkdir(exist_ok=True)
    binary = build_dir / f"task{task}_test"
    binary.unlink(missing_ok=True)  # 不运行上次编译留下的可执行文件。
    command = ["g++", "-std=c++17", "-O2", "-Wall", "-Wextra", "-pedantic",
               "-I", str(PRACTICE_DIR / "cpp")]
    command += [str(PRACTICE_DIR / "cpp" / name) for name in sources]
    subprocess.run(command + ["-o", str(binary)], check=True)
    subprocess.run([str(binary)], cwd=PRACTICE_DIR, check=True)

print("实践目录：", PRACTICE_DIR)


### 0.2 输入数据

本实践使用 262144 个顶点，编号为 0～262143。

`ExerciseConfig` 中已经配置好实验中所需的随机种子等实验参数。


---
## 1. 任务一：数据结构的定义

邻接表由顶点表和边结点组成。`VNode.data` 保存整数编号，`firstarc` 指向该顶点的第一条邻接边；`ArcNode.adjvex` 保存邻接顶点的位置，`nextarc` 指向下一条邻接边，`info=1` 表示一跳连接。`ALGraph` 保存顶点表 `vertices`、实际顶点数 `vexnum` 和无向边数 `arcnum`。

请补全：

- `TODO 1A`：在 `ArcNode` 中定义指针成员 `nextarc`，使它能指向另一个边结点；
- `TODO 1B`：在 `VNode` 中定义指针成员 `firstarc`，使它能指向该顶点的首个边结点。

这两个指针共同建立邻接关系：从 `vertices[u].firstarc` 找到首个边结点，再沿 `nextarc` 依次访问后续结点，直到空指针。顶点编号从 0 开始，`data` 与顶点在表中的位置相同。

本节只定义邻接表的数据结构。完成填空并运行下面的代码单元后，在任务二的前置区域运行结构检查单元，确认成员定义正确后再继续构图。


In [ ]:
%%writefile Source/02.06/cpp/practice_types.hpp
#pragma once

#define MVNum 262144

// 边结点通过 nextarc 组成单链表，adjvex 保存邻接顶点的位置。
typedef struct ArcNode {
    int adjvex;
    // TODO 1A：定义指向下一条边的指针成员 nextarc。

    int info;
} ArcNode;

typedef struct VNode {
    int data;
    // TODO 1B：定义指向该顶点首个边结点的指针成员 firstarc。

} VNode, AdjList[MVNum];

typedef struct {
    AdjList vertices;
    int vexnum, arcnum;
} ALGraph;


---
## 2. 任务二：图的构建

任务一完成了邻接表的成员定义。现在使用头插法，把每条邻接记录连接到对应顶点的链表中。

以下前置代码提供实验参数、配置检查、空图初始化、内存释放以及网络和节点状态生成等辅助功能，不需要补全。先依次运行前置代码和结构检查单元，确认任务一的成员类型、顶点表容量及空邻接链初始化正确，再完成头插操作。

辅助函数 `openTopology` 在堆上分配顶点表，将实际顶点的 `data` 设为编号、`firstarc` 设为空指针，并将无向边数初始化为 0。所有权和释放代码已经提供，不要对含裸指针的 `ALGraph` 做按值拷贝。


In [ ]:
%%writefile Source/02.06/cpp/practice_graph_support.hpp
#pragma once
#include "practice_types.hpp"

#include <algorithm>
#include <chrono>
#include <cstdint>
#include <cstdlib>
#include <iomanip>
#include <fstream>
#include <iostream>
#include <limits>
#include <numeric>
#include <memory>
#include <stdexcept>
#include <string>
#include <vector>

#define OK 1

constexpr int32_t kUnvisited = -1;
constexpr int32_t kNoCandidate = -1;
constexpr int32_t kDeviceInvalidCandidate = 0x7fffffff;
constexpr uint32_t kStaticMaxFrontierBlockLen = 2048;
constexpr uint32_t kStaticMaxNeighborTileLen = 2048;

struct ExerciseConfig {
    uint32_t vertexCount = MVNum;
    uint32_t degreeTarget = 32;
    uint32_t randomSeed = 1234;
    uint32_t origin = 0;
    uint32_t healthyRate = 1;
    // Nodes with BFS distance <= fault_radius from fault_node are forced unavailable.
    // This creates deterministic multi-level diagnosis cases for performance tests.
    uint32_t blockedHops = 4;
    uint32_t frontierChunk = 32;
    uint32_t neighborChunk = 32;
    uint32_t tasks = 16;
    uint32_t preheat = 2;
    uint32_t trials = 10;
    uint32_t depthBound = 0;  // 0 means num_nodes.
    std::string csvFile;
    bool sweep = false;
    bool showGraph = false;
    bool showLevel = false;
};

inline uint32_t depthCeiling(const ExerciseConfig& options) {
    return options.depthBound == 0 ? options.vertexCount : options.depthBound;
}

inline uint32_t alignElements(uint32_t x, uint32_t align) {
    if (align == 0) return x;
    return ((x + align - 1) / align) * align;
}

inline void verifySettings(const ExerciseConfig& options) {
    if (options.vertexCount == 0 || options.vertexCount > MVNum) {
        throw std::invalid_argument("--nodes must be in [1, MVNum]");
    }
    if (options.origin >= options.vertexCount) {
        throw std::invalid_argument("--fault-node must be in [0, nodes)");
    }
    if (options.degreeTarget == 0) {
        throw std::invalid_argument("--avg-degree must be positive");
    }
    if (options.healthyRate > 100) {
        throw std::invalid_argument("--health-percent must be in [0, 100]");
    }
    if (options.frontierChunk == 0 || options.frontierChunk > kStaticMaxFrontierBlockLen) {
        throw std::invalid_argument("--frontier-block-len must be in [1, 2048]");
    }
    if (options.neighborChunk == 0 || options.neighborChunk > kStaticMaxNeighborTileLen) {
        throw std::invalid_argument("--neighbor-tile-len must be in [1, 2048]");
    }
    if ((options.frontierChunk * sizeof(int32_t)) % 32 != 0) {
        throw std::invalid_argument("frontier_block_len*sizeof(int32_t) must be 32B aligned");
    }
    if ((options.neighborChunk * sizeof(int32_t)) % 32 != 0) {
        throw std::invalid_argument("neighbor_tile_len*sizeof(int32_t) must be 32B aligned");
    }
    if (options.trials == 0) throw std::invalid_argument("--repeat must be positive");
    if (options.tasks == 0) {
        throw std::invalid_argument("--block-dim must be positive");
    }
}


#include <deque>
#include <random>
#include <unordered_set>
#include <utility>

inline void closeTopology(ALGraph* net) noexcept;
using TopologyPtr = std::unique_ptr<ALGraph, decltype(&closeTopology)>;
struct TopologyInput {
    TopologyPtr links;
    std::vector<int32_t> availability;
};

inline void closeTopology(ALGraph* net) noexcept {
    if (!net) return;
    for (int from = 0; from < net->vexnum; ++from) {
        ArcNode* cursor = net->vertices[from].firstarc;
        while (cursor != nullptr) {
            ArcNode* upcoming = cursor->nextarc;
            delete cursor;
            cursor = upcoming;
        }
        net->vertices[from].firstarc = nullptr;
    }
    net->vexnum = net->arcnum = 0;
    delete net;
}

inline TopologyPtr openTopology(uint32_t count) {
    if (count == 0 || count > MVNum)
        throw std::invalid_argument("vertex count exceeds MVNum");
    // 顶点表放在堆上，避免大型数组占用函数栈。
    TopologyPtr net(new ALGraph{}, closeTopology);
    net->vexnum = static_cast<int>(count);
    for (int i = 0; i < net->vexnum; ++i) {
        net->vertices[i].data = i;
        net->vertices[i].firstarc = nullptr;
    }
    return net;
}

void attachNeighbor(VNode& anchor, ArcNode* entry);

inline int joinEndpoints(ALGraph& net, int from, int to) {
    if (from < 0 || to < 0 || from >= net.vexnum || to >= net.vexnum)
        throw std::invalid_argument("edge endpoint is outside the graph");
    if (from == to) return 0;
    if (net.arcnum >= std::numeric_limits<int32_t>::max() / 2)
        throw std::overflow_error("too many adjacency entries");
    std::unique_ptr<ArcNode> out(new ArcNode{to, nullptr, 1});
    std::unique_ptr<ArcNode> back(new ArcNode{from, nullptr, 1});
    attachNeighbor(net.vertices[from], out.get());
    attachNeighbor(net.vertices[to], back.get());
    out.release();
    back.release();
    ++net.arcnum;
    return OK;
}

namespace {
inline uint64_t connectionKey(uint32_t from, uint32_t to) {
    if (from > to) std::swap(from, to);
    return (static_cast<uint64_t>(from) << 32) | static_cast<uint64_t>(to);
}

inline void includeLink(uint32_t from, uint32_t to, ALGraph& net,
                     std::unordered_set<uint64_t>& seen) {
    if (from == to || !seen.insert(connectionKey(from, to)).second) return;
    joinEndpoints(net, static_cast<int>(from), static_cast<int>(to));
}

// 用于准备节点状态，距离只计算到给定半径。
inline std::vector<int32_t> radiusLabels(const ALGraph& net,
                                                uint32_t source, uint32_t radius) {
    std::vector<int32_t> round(net.vexnum, -1);
    std::deque<uint32_t> q;
    round[source] = 0;
    q.push_back(source);
    while (!q.empty()) {
        const uint32_t from = q.front();
        q.pop_front();
        if (static_cast<uint32_t>(round[from]) >= radius) continue;
        for (const ArcNode* cursor = net.vertices[from].firstarc; cursor != nullptr; cursor = cursor->nextarc) {
            const uint32_t to = static_cast<uint32_t>(cursor->adjvex);
            if (round[to] >= 0) continue;
            round[to] = round[from] + 1;
            q.push_back(to);
        }
    }
    return round;
}
}  // namespace

inline TopologyInput prepareTopology(const ExerciseConfig& options) {
    verifySettings(options);
    const uint32_t vertexTotal = options.vertexCount;
    auto net = openTopology(vertexTotal);
    std::unordered_set<uint64_t> seen;
    seen.reserve(static_cast<size_t>(vertexTotal) * std::max(2u, options.degreeTarget));
    if (vertexTotal >= 2) {
        for (uint32_t from = 0; from < vertexTotal; ++from)
            includeLink(from, (from + 1) % vertexTotal, *net, seen);
    }
    const uint64_t wanted = std::max<uint64_t>(seen.size(),
        std::max<uint64_t>(1, static_cast<uint64_t>(vertexTotal) * options.degreeTarget / 2));
    std::mt19937 engine(options.randomSeed);
    std::uniform_int_distribution<uint32_t> node_dist(0, vertexTotal - 1);
    uint64_t attempts = 0;
    const uint64_t attemptLimit = wanted * 20 + 1024;
    while (seen.size() < wanted && attempts++ < attemptLimit) {
        const uint32_t from = node_dist(engine);
        uint32_t to = node_dist(engine);
        if (from == to) to = (to + 1) % vertexTotal;
        includeLink(from, to, *net, seen);
    }
    std::vector<int32_t> status(vertexTotal, 0);
    std::uniform_int_distribution<uint32_t> pct_dist(1, 100);
    for (uint32_t from = 0; from < vertexTotal; ++from) status[from] = pct_dist(engine) <= options.healthyRate ? 1 : 0;
    std::vector<int32_t> near_fault;
    if (options.blockedHops > 0) {
        near_fault = radiusLabels(*net, options.origin, options.blockedHops);
        for (uint32_t from = 0; from < vertexTotal; ++from)
            if (near_fault[from] >= 0) status[from] = 0;
    } else {
        status[options.origin] = 0;
    }
    if (std::find(status.begin(), status.end(), 1) == status.end() && vertexTotal > 1) {
        if (near_fault.empty())
            near_fault = radiusLabels(*net, options.origin, options.blockedHops);
        for (uint32_t step = 1; step < vertexTotal; ++step) {
            const uint32_t candidate = (options.origin + step) % vertexTotal;
            if (candidate != options.origin && (options.blockedHops == 0 || near_fault[candidate] < 0)) {
                status[candidate] = 1;
                break;
            }
        }
    }
    return TopologyInput{std::move(net), std::move(status)};
}



In [ ]:
%%writefile Source/02.06/cpp/task1_test.cpp
#include "practice_graph_support.hpp"
#include <cassert>
#include <type_traits>

int main() {
    static_assert(std::is_same_v<decltype(ArcNode{}.adjvex), int>);
    static_assert(std::is_same_v<decltype(ArcNode{}.nextarc), ArcNode*>);
    static_assert(std::is_same_v<decltype(ArcNode{}.info), int>);
    static_assert(std::is_same_v<decltype(VNode{}.data), int>);
    static_assert(std::is_same_v<decltype(VNode{}.firstarc), ArcNode*>);
    static_assert(std::extent_v<AdjList> == MVNum);
    static_assert(std::is_same_v<decltype(ALGraph{}.vertices), AdjList>);
    static_assert(std::is_same_v<decltype(ALGraph{}.vexnum), int>);
    static_assert(std::is_same_v<decltype(ALGraph{}.arcnum), int>);

    const ExerciseConfig options;
    auto net = openTopology(options.vertexCount);
    assert(net->vexnum == static_cast<int>(options.vertexCount));
    assert(net->arcnum == 0);
    for (int from = 0; from < net->vexnum; ++from) {
        assert(net->vertices[from].data == from);
        assert(net->vertices[from].firstarc == nullptr);
    }
    std::cout << "vertices=" << net->vexnum << " undirected_edges=" << net->arcnum
              << " capacity=" << MVNum << '\n';
    std::cout << "Task 1 PASSED\n";
}


In [ ]:
test_cpp(1, ["task1_test.cpp"])


`attachNeighbor` 接收顶点 `anchor` 和已经分配好的边结点 `entry`。

请补全：

- `TODO 2A`：先让 `entry` 的后继指向 `anchor` 原来的首边，再更新 `anchor.firstarc`，使 `entry` 成为新的首边结点。


In [ ]:
%%writefile Source/02.06/cpp/practice_graph.cpp
#include "practice_graph_support.hpp"

void attachNeighbor(VNode& anchor, ArcNode* entry) {
    // TODO 2A：先让 entry 连接原首结点，再把它设为 anchor 的首结点。

}


In [ ]:
%%writefile Source/02.06/cpp/task2_test.cpp
#include "practice_graph_support.hpp"
#include <cassert>
#include <type_traits>

int main() {
    static_assert(std::is_same_v<decltype(VNode{}.data), int>);
    static_assert(std::is_same_v<decltype(ArcNode{}.nextarc), ArcNode*>);
    // 观察连续插入后的遍历顺序，检查空链表和已有结点两种情况。
    VNode probe{0, nullptr};
    ArcNode records[] = {{7, nullptr, 1}, {11, nullptr, 1}, {13, nullptr, 1}};
    for (ArcNode& entry : records) attachNeighbor(probe, &entry);
    const int expected[] = {13, 11, 7};
    const ArcNode* cursor = probe.firstarc;
    for (int adjacent : expected) {
        assert(cursor != nullptr && cursor->adjvex == adjacent);
        cursor = cursor->nextarc;
    }
    assert(cursor == nullptr);

    const ExerciseConfig options;
    auto input = prepareTopology(options);
    const ALGraph& net = *input.links;
    assert(net.vexnum == static_cast<int>(options.vertexCount));
    uint64_t entries = 0;
    for (int from = 0; from < net.vexnum; ++from) {
        assert(net.vertices[from].data == from);
        for (const ArcNode* cursor = net.vertices[from].firstarc; cursor; cursor = cursor->nextarc) {
            assert(cursor->adjvex >= 0 && cursor->adjvex < net.vexnum);
            assert(cursor->adjvex != from && cursor->info == 1);
            assert(++entries <= static_cast<uint64_t>(net.arcnum) * 2);
            if (from < 16) {
                bool reverseFound = false;
                for (const ArcNode* back = net.vertices[cursor->adjvex].firstarc; back; back = back->nextarc)
                    if (back->adjvex == from) reverseFound = true;
                assert(reverseFound);
            }
        }
    }
    assert(entries == static_cast<uint64_t>(net.arcnum) * 2);
    assert(input.availability.size() == static_cast<size_t>(net.vexnum));
    std::cout << "vertices=" << net.vexnum << " undirected_edges=" << net.arcnum
              << " adjacency_entries=" << entries << '\n';
    for (int from = 0; from < 4; ++from) {
        std::cout << "data=" << net.vertices[from].data << ": ";
        int shown = 0;
        for (const ArcNode* cursor = net.vertices[from].firstarc; cursor && shown < 12;
             cursor = cursor->nextarc, ++shown) std::cout << cursor->adjvex << ' ';
        std::cout << "...\n";
    }
    input.links.reset();
    std::cout << "Task 2 PASSED\n";
}


In [ ]:
test_cpp(2, ["practice_graph.cpp", "task2_test.cpp"])


---
## 3. 任务三：图的 CSR 压缩存储

本任务沿链表把各顶点的邻接段组织为连续数组。`PracticeCsr` 保存实际顶点数 `vertexCount`、行边界 `rowBegin`、邻接编号 `neighbor` 和可用状态 `available`。它们分别对应讲解中的 CSR 顶点数、偏移数组、邻居数组和状态数组。

### 3.1 记录各段边界

`rowBegin` 长度为顶点数加 1，首项为 0。第一遍遍历完成一个顶点后，累计记录数就是该邻接段的结束边界。

请补全：
- `TODO 3A`：把 `adjacencyCount` 写入 `packed.rowBegin[from+1]`。


顶点 u 的邻居位于 `[rowBegin[u], rowBegin[u+1])`。没有邻居时两个边界相等，末项应等于 `2*arcnum`。测试逐顶点比较链表中的邻居与 CSR 邻接段；转换结束后，辅助函数释放链式图。

### 3.2 连续保存邻接顶点

`packAdjacency` 先遍历各顶点的链表，累计边结点总数；随后分配连续数组，再沿相同的指针路径填写邻接位置。

请补全：
- `TODO 3B`：把 `cursor->adjvex` 写入 `packed.neighbor[writeAt]`，并将写入位置加一。

每个邻接段填完后会按编号排序，因此头插法造成的链表顺序不影响 CSR 的确定性。转换保留全部邻接关系，包括无向边的两个方向。

In [ ]:
%%writefile Source/02.06/cpp/practice_build.hpp
#pragma once
#include "practice_graph_support.hpp"

struct PracticeCsr {
    uint32_t vertexCount = 0;
    uint32_t edgePairs = 0;
    std::vector<int32_t> rowBegin;
    std::vector<int32_t> neighbor;
    std::vector<int32_t> available;  // 1 means healthy and replaceable; 0 means faulty/unavailable.
};


PracticeCsr packAdjacency(const ALGraph& net, const std::vector<int32_t>& available);
PracticeCsr assembleNetwork(const ExerciseConfig& options);


In [ ]:
%%writefile Source/02.06/cpp/practice_build.cpp
#include "practice_build.hpp"

PracticeCsr packAdjacency(const ALGraph& net, const std::vector<int32_t>& available) {
    if (net.vexnum <= 0 || net.vexnum > MVNum || net.arcnum < 0 ||
        available.size() != static_cast<size_t>(net.vexnum))
        throw std::invalid_argument("invalid ALGraph or vertex available");
    PracticeCsr packed;
    packed.vertexCount = static_cast<uint32_t>(net.vexnum);
    packed.edgePairs = static_cast<uint32_t>(net.arcnum);
    packed.rowBegin.assign(static_cast<size_t>(net.vexnum) + 1, 0);
    uint64_t adjacencyCount = 0;
    // 第一遍：沿链表计数，以累计长度记录各邻接段边界。
    for (int from = 0; from < net.vexnum; ++from) {
        if (net.vertices[from].data != from || (available[from] != 0 && available[from] != 1))
            throw std::invalid_argument("invalid vertex data or available");
        for (const ArcNode* cursor = net.vertices[from].firstarc; cursor != nullptr; cursor = cursor->nextarc) {
            if (cursor->adjvex < 0 || cursor->adjvex >= net.vexnum || cursor->info != 1)
                throw std::invalid_argument("invalid adjacency node");
            if (++adjacencyCount > static_cast<uint64_t>(net.arcnum) * 2 ||
                adjacencyCount > static_cast<uint64_t>(std::numeric_limits<int32_t>::max()))
                throw std::runtime_error("adjacency count exceeds graph metadata");
        }
        // TODO 3A：将累计邻接记录数 adjacencyCount 写入下一项行边界。


    }
    if (adjacencyCount != static_cast<uint64_t>(net.arcnum) * 2)
        throw std::runtime_error("each undirected edge must have two adjacency nodes");
    // 第二遍：将边结点中的邻接位置写入连续数组。
    packed.neighbor.resize(static_cast<size_t>(adjacencyCount));
    for (int from = 0; from < net.vexnum; ++from) {
        int32_t writeAt = packed.rowBegin[from];
        for (const ArcNode* cursor = net.vertices[from].firstarc; cursor != nullptr; cursor = cursor->nextarc)
        {
            // TODO 3B：把当前边结点的邻接位置写入 packed.neighbor[writeAt]，并递增 writeAt。

        }
        std::sort(packed.neighbor.begin() + packed.rowBegin[from], packed.neighbor.begin() + packed.rowBegin[from + 1]);
    }
    packed.available = available;
    return packed;
}

PracticeCsr assembleNetwork(const ExerciseConfig& options) {
    auto input = prepareTopology(options);
    auto packed = packAdjacency(*input.links, input.availability);
    input.links.reset();
    return packed;
}


In [ ]:
%%writefile Source/02.06/cpp/task3_test.cpp
#include "practice_build.hpp"
#include <cassert>

int main() {
    const ExerciseConfig options;
    auto input = prepareTopology(options);
    const ALGraph& net = *input.links;
    const auto packed = packAdjacency(net, input.availability);
    assert(packed.vertexCount == static_cast<uint32_t>(net.vexnum));
    assert(packed.available == input.availability);
    assert(packed.rowBegin.size() == static_cast<size_t>(net.vexnum) + 1);
    assert(packed.rowBegin.front() == 0);
    assert(packed.neighbor.size() == static_cast<size_t>(net.arcnum) * 2);
    assert(packed.rowBegin.back() == static_cast<int32_t>(packed.neighbor.size()));
    for (int from = 0; from < net.vexnum; ++from) {
        std::vector<int32_t> expected;
        for (const ArcNode* cursor = net.vertices[from].firstarc; cursor; cursor = cursor->nextarc)
            expected.push_back(cursor->adjvex);
        std::sort(expected.begin(), expected.end());
        const auto first = packed.neighbor.begin() + packed.rowBegin[from];
        const auto last = packed.neighbor.begin() + packed.rowBegin[from + 1];
        assert(std::equal(expected.begin(), expected.end(), first, last));
    }
    std::cout << "rowBegin length=" << packed.rowBegin.size()
              << " neighbor length=" << packed.neighbor.size() << '\n';
    for (uint32_t from = 0; from < 4; ++from)
        std::cout << "vertex " << from << " range=[" << packed.rowBegin[from]
                  << ',' << packed.rowBegin[from + 1] << ")\n";
    std::cout << "Task 3 PASSED\n";
}


In [ ]:
test_cpp(3, ["practice_graph.cpp", "practice_build.cpp", "task3_test.cpp"])


---
## 4. 任务四：基于 Ascend C 实现并行 BFS

整体过程：
1. Host 保存当前层的顶点，将它们按前沿块分配给 16 个 NPU 任务。
2. 每个任务根据 CSR 边界找到这些顶点的邻居，分段搬入局部张量 tile。
3. 任务读取邻居编号，在自己的位图中标记“本层遇到了这个顶点”。
4. Host 合并各任务的位图，过滤已经访问过的顶点，记录层号并构造下一层。
5. 如果这一层出现可用替代节点，就选择其中编号最小者并结束；否则继续扩展下一层。

### 4.1 数据表示

| 数据 | Host 侧 | NPU 侧 |
|---|---|---|
| 邻接段边界 | `rowBegin` | `rowGm` |
| 连续邻接编号 | `neighbor` | `columnGm`，分段搬入 UB 的 `tile` |
| 当前层顶点 | `wave` | `frontierGm`，分块搬入 UB 的 `batch` |
| 发现结果 | 合并后的位图 | 各任务独立的 `outputGm` 区域 |

程序默认启动 16 个任务，按连续前沿块分工。每个顶点的邻接段由领取它的任务遍历。前沿很小时，部分任务可能没有工作；任务数不等同于设备物理核心数。

一个 32 位整数记录 32 个顶点。对邻居 `adjacent`，通过除以 32 确定所在字，通过余数确定该字中的位。各任务的整份位图按 64 字节对齐，重复标记同一顶点不会增加记录数量。

Host 等待整层 Kernel 完成，合并各核位图，排除已访问顶点，记录新层号并构造下一层。候选选择也在整层合并完成后进行。


### 4.2 确定 CSR 邻居数组边界

顶点 `u` 的邻居在 CSR 邻居数组中的位置为 `[rowBegin[u], rowBegin[u+1])`。在 NPU 侧，行边界数组通过全局张量 `rowGm` 访问。

请补全：
- `TODO 4A`：读取顶点 `u` 的两个行边界，分别赋给 `begin` 和 `end`。参数 `begin`、`end` 是引用，赋值后结果会传回调用处。

**Ascend C 接口提示：** `GlobalTensor<int32_t>` 表示绑定到全局内存的张量；`rowGm.GetValue(index)` 读取指定下标处的一个整数。

**注：** 函数前的 `__aicore__` 修饰符已给出，用于声明在 NPU 上调用的函数，无需修改。


In [ ]:
%%writefile Source/02.06/ascend_ops/op_kernel/neighbor_range.hpp
#pragma once
#include "kernel_operator.h"

__aicore__ inline void ReadNeighborRange(
    const AscendC::GlobalTensor<int32_t>& rowGm,
    uint32_t u,int32_t& begin,int32_t& end) {
    // TODO 4A：读取顶点 u 的邻接段起止边界，分别赋给 begin 和 end。


}


In [ ]:
%%writefile Source/02.06/ascend_ops/op_kernel/neighbor_expand.cpp
// Experiment 2 kernel: one BFS level expansion over a CSR graph.
//
// Host-merged per-core bitset design:
//   - current frontier is stored as a dense int32 array in GM;
//   - CSR row_ptr and col_idx encode adjacency lists;
//   - each AI Core owns a contiguous range of frontier blocks;
//   - static UB tensors stage frontier nodes and neighbor segments;
//   - the kernel writes a per-core bitset slice for discovered neighbor
//     vertices, using one bit per graph vertex;
//   - Host OR-reduces the per-core bitsets and performs the canonical
//     visited/level update.
//
// Private per-core bitsets provide race-free discovery while reducing the
// returned next-frontier state from blockDim*numNodes int32 flags to
// blockDim*round_up(ceil(numNodes/32),16) int32 bitset words.

#include "kernel_operator.h"
#include "neighbor_range.hpp"

using namespace AscendC;

namespace {
constexpr uint32_t kStaticMaxFrontierBlockLen = 2048;
constexpr uint32_t kStaticMaxNeighborTileLen = 2048;

__aicore__ inline void AssignChunkRange(uint32_t numBlocks,
                                      uint32_t taskCount,
                                      uint32_t& firstChunk,
                                      uint32_t& lastChunk) {
    const uint32_t workerCount = taskCount == 0 ? 1 : taskCount;
    const uint32_t taskId = GetBlockIdx();
    if (taskId >= workerCount) {
        firstChunk = 0;
        lastChunk = 0;
        return;
    }
    const uint32_t blocksPerCore = (numBlocks + workerCount - 1) / workerCount;
    firstChunk = taskId * blocksPerCore;
    lastChunk = firstChunk + blocksPerCore;
    if (lastChunk > numBlocks) {
        lastChunk = numBlocks;
    }
}

__aicore__ inline uint32_t Shorter(uint32_t a, uint32_t b) {
    return a < b ? a : b;
}
}  // namespace

extern "C" __global__ __aicore__ void neighbor_expand(GM_ADDR row,
                                                        GM_ADDR column,
                                                        GM_ADDR frontier,
                                                        GM_ADDR output,
                                                        uint32_t vertexCount,
                                                        uint32_t edgeCapacity,
                                                        uint32_t activeCount,
                                                        uint32_t frontierCapacity,
                                                        uint32_t frontierChunk,
                                                        uint32_t neighborChunk,
                                                        uint32_t taskCount) {
    InitSocState();

    GlobalTensor<int32_t> rowGm;
    GlobalTensor<int32_t> columnGm;
    GlobalTensor<int32_t> frontierGm;
    GlobalTensor<int32_t> outputGm;

    rowGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(row), vertexCount + 1);
    columnGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(column), edgeCapacity);
    frontierGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(frontier), frontierCapacity);
    const uint32_t wordsPerBitmap = (((vertexCount + 31u) >> 5) + 15u) / 16u * 16u;
    outputGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(output), wordsPerBitmap * taskCount);

    const uint32_t taskId = GetBlockIdx();
    if (taskId >= taskCount) {
        return;
    }

    if (frontierChunk == 0 || frontierChunk > kStaticMaxFrontierBlockLen ||
        neighborChunk == 0 || neighborChunk > kStaticMaxNeighborTileLen ||
        activeCount == 0 || vertexCount == 0 || edgeCapacity == 0) {
        return;
    }

    const uint32_t chunkCount = (activeCount + frontierChunk - 1) / frontierChunk;
    uint32_t firstChunk = 0;
    uint32_t lastChunk = 0;
    AssignChunkRange(chunkCount, taskCount, firstChunk, lastChunk);
    if (firstChunk >= lastChunk) {
        return;
    }

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<int32_t> batch = ubAllocator.Alloc<int32_t, kStaticMaxFrontierBlockLen>();
    LocalTensor<int32_t> tile = ubAllocator.Alloc<int32_t, kStaticMaxNeighborTileLen>();
    batch.SetSize(kStaticMaxFrontierBlockLen);
    tile.SetSize(kStaticMaxNeighborTileLen);


    for (uint32_t chunk = firstChunk; chunk < lastChunk; ++chunk) {
        const uint32_t batchStart = chunk * frontierChunk;
        const uint32_t batchLength = Shorter(frontierChunk, activeCount - batchStart);
        DataCopy(batch, frontierGm[batchStart], frontierChunk);
        PipeBarrier<PIPE_ALL>();

        for (uint32_t item = 0; item < batchLength; ++item) {
            const int32_t vertex = batch.GetValue(item);
            if (vertex < 0 || static_cast<uint32_t>(vertex) >= vertexCount) {
                continue;
            }
            int32_t begin = 0, end = 0;
            ReadNeighborRange(rowGm, static_cast<uint32_t>(vertex), begin, end);
            if (end <= begin) {
                continue;
            }

            uint32_t edgeOffset = static_cast<uint32_t>(begin);
            const uint32_t edgeLimit = static_cast<uint32_t>(end);
            while (edgeOffset < edgeLimit) {
                const uint32_t tileLength = Shorter(neighborChunk, edgeLimit - edgeOffset);
                DataCopy(tile, columnGm[edgeOffset], neighborChunk);
                PipeBarrier<PIPE_ALL>();

                for (uint32_t slot = 0; slot < tileLength; ++slot) {
                    int32_t adjacent = -1;
                    adjacent = tile.GetValue(slot);
                    if (adjacent < 0 || static_cast<uint32_t>(adjacent) >= vertexCount) {
                        continue;
                    }
                    // Per-core bitset layout:
                    // nextBitmap[coreId * wordsPerBitmap + (vertex / 32)].
                    // Each AI Core owns a private slice, so there is no cross-core
                    // write race. Host later OR-reduces these slices and assigns the
                    // canonical BFS level for the next frontier.
                    const uint32_t targetIndex = static_cast<uint32_t>(adjacent);
                    const uint32_t wordOffset = targetIndex >> 5;
                    const uint32_t mask = 1u << (targetIndex & 31u);
                    const uint32_t wordIndex = taskId * wordsPerBitmap + wordOffset;
                    uint32_t found = static_cast<uint32_t>(outputGm.GetValue(wordIndex));
                    found = found | mask;
                    outputGm.SetValue(wordIndex, static_cast<int32_t>(found));
                }
                edgeOffset += tileLength;
            }
        }
    }

    // 将本核位图写回 GM，并使后续层不再使用旧缓存。
    DataCacheCleanAndInvalid<int32_t, CacheLine::ENTIRE_DATA_CACHE,
                             DcciDst::CACHELINE_OUT>(outputGm);
}


### 4.3 排除已访问顶点，构造下一层

Host 已将各任务的位图合并。下面的 `buildNextFrontier` 从合并结果中依次取出顶点编号 `to`，检查这些顶点能否进入下一层。

请补全：
- `TODO 4B`：如果 `to` 超出顶点范围，或 `visitedHost[to]` 表示该顶点已经访问过，就使用 `continue` 跳过当前顶点。

其中，`vertexTotal` 是实际顶点数，`visitedHost[to]` 为 0 表示尚未访问，为 1 表示已经访问。检查编号范围时，应先判断 `to` 是否有效，再访问数组元素。

**注：** 位图解码、设置访问标记、记录层号、加入 `upcoming` 以及选择本层最小候选的代码均已提供。本题只需补全标准 C++ 的条件判断与跳过操作。


In [ ]:
%%writefile Source/02.06/cpp/practice_frontier.hpp
#pragma once
#include "practice_build.hpp"

inline int32_t buildNextFrontier(const PracticeCsr& graph,
                                uint32_t origin, uint32_t round,
                                const std::vector<uint32_t>& mergedBitmap,
                                std::vector<int32_t>& visitedHost,
                                std::vector<int32_t>& levelHost,
                                std::vector<int32_t>& upcoming) {
    const uint32_t vertexTotal = graph.vertexCount;
    int32_t best = kDeviceInvalidCandidate;
    for (uint32_t w = 0; w < mergedBitmap.size(); ++w) {
        uint32_t word = mergedBitmap[w];
        while (word != 0u) {
            // 已提供：从位图中取得一个待检查的顶点编号。
            const uint32_t bit = static_cast<uint32_t>(__builtin_ctz(word));
            const uint32_t to = (w << 5) + bit;
            word &= (word - 1u);

            // TODO 4B：跳过编号越界或已经访问过的顶点。


            visitedHost[to] = 1;
            levelHost[to] = static_cast<int32_t>(round + 1);
            upcoming.push_back(static_cast<int32_t>(to));
            if (to != origin && graph.available[to] == 1 &&
                static_cast<int32_t>(to) < best) {
                best = static_cast<int32_t>(to);
            }
        }
    }
    return best;
}


### 4.4 CPU 串行参考实现

下面给出完整的结果结构、辅助接口和 CPU 串行 BFS，不需要补全。串行实现使用与 NPU 相同的 CSR、源点、节点状态和候选选择规则，供后续程序核对候选编号、距离与全部层号。


In [ ]:
%%writefile Source/02.06/cpp/practice_search.hpp
#pragma once
#include "practice_build.hpp"

struct SearchTiming {
    double preprocess_us = 0.0;
    double total_us = 0.0;
    double kernel_us = 0.0;
    double host_frontier_us = 0.0;
};

struct SearchState {
    int32_t replacement = kNoCandidate;
    int32_t distance = -1;
    uint32_t visited_count = 0;
    uint64_t processed_edges = 0;
    std::vector<int32_t> level;
    std::vector<uint32_t> waveSizes;
    std::vector<uint64_t> waveEdges;
    SearchTiming timing;
};

struct ResultCheck {
    bool replacement_ok = false;
    bool distance_ok = false;
    bool level_ok = false;
    uint32_t level_mismatch = 0;
};

class Stopwatch {
public:
    Stopwatch() : start_(std::chrono::high_resolution_clock::now()) {}
    double elapsed_us() const {
        const auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(end - start_).count();
    }
private:
    std::chrono::high_resolution_clock::time_point start_;
};

inline uint64_t adjacencyEntries(const PracticeCsr& packed) {
    return static_cast<uint64_t>(packed.neighbor.size());
}

inline double edgeRate(uint64_t edges, double us) {
    if (us <= 0.0) return 0.0;
    return static_cast<double>(edges) / (us * 1e-6);
}

inline ResultCheck checkSearch(const SearchState& got, const SearchState& ref) {
    ResultCheck c;
    c.replacement_ok = (got.replacement == ref.replacement);
    c.distance_ok = (got.distance == ref.distance);
    c.level_ok = (got.level.size() == ref.level.size());
    if (c.level_ok) {
        for (size_t i = 0; i < got.level.size(); ++i) {
            if (got.level[i] != ref.level[i]) {
                ++c.level_mismatch;
                c.level_ok = false;
            }
        }
    }
    return c;
}

inline void describeNetwork(const PracticeCsr& packed, uint32_t origin) {
    uint32_t healthy = 0;
    for (int32_t s : packed.available) healthy += (s == 1 ? 1u : 0u);
    std::cout << "graph: nodes=" << packed.vertexCount
              << ", directed_edges=" << packed.neighbor.size()
              << ", undirected_edges=" << packed.edgePairs
              << ", healthy_nodes=" << healthy
              << ", fault_node=" << origin << "\n";
}

inline void showLayers(const SearchState& state, size_t count = 32) {
    const size_t vertexTotal = std::min(state.level.size(), count);
    std::cout << "level sample(first " << vertexTotal << "):\n";
    for (size_t i = 0; i < vertexTotal; ++i) {
        std::cout << "  node=" << i << " level=" << state.level[i] << "\n";
    }
}

inline void showTimingHeader(bool) {
    std::cout << std::setw(14) << "method" << std::setw(16) << "avg_us"
              << std::setw(14) << "status" << '\n';
}
inline void showTimingRow(const PracticeCsr&, uint32_t, const SearchState& state,
                             const std::string& status, bool ascend) {
    std::cout << std::setw(14) << (ascend ? "NPU_BFS" : "CPU_BFS")
              << std::setw(16) << std::fixed << std::setprecision(3)
              << state.timing.total_us << std::setw(14) << status << '\n';
}
inline void saveTimingCsv(const std::string& path, const std::string& method,
                             double avg_us, const std::string& status, bool append=false) {
    if (path.empty()) return;
    std::ofstream out(path, append ? std::ios::app : std::ios::out);
    if (!out) throw std::runtime_error("cannot write CSV: " + path);
    if (!append) out << "method,avg_us,status\n";
    out << method << ',' << std::fixed << std::setprecision(3)
        << avg_us << ',' << status << '\n';
}

SearchState searchSerial(const PracticeCsr& packed, uint32_t origin, uint32_t depthBound = 0);


In [ ]:
%%writefile Source/02.06/cpp/practice_search.cpp
#include "practice_search.hpp"

SearchState searchSerial(const PracticeCsr& packed, uint32_t origin, uint32_t depthBound) {
    if (origin >= packed.vertexCount) {
        throw std::invalid_argument("fault_node must be in [0, num_nodes)");
    }
    if (packed.rowBegin.size() != static_cast<size_t>(packed.vertexCount) + 1 ||
        packed.available.size() != packed.vertexCount) {
        throw std::invalid_argument("invalid CSR graph");
    }
    const uint32_t limit = depthBound == 0 ? packed.vertexCount : depthBound;

    SearchState state;
    state.level.assign(packed.vertexCount, kUnvisited);
    std::vector<int32_t> seenVertices(packed.vertexCount, 0);
    std::vector<int32_t> wave;
    wave.reserve(packed.vertexCount);
    wave.push_back(static_cast<int32_t>(origin));
    seenVertices[origin] = 1;
    state.level[origin] = 0;
    state.visited_count = 1;

    Stopwatch watch;
    for (uint32_t round = 0; !wave.empty() && round < limit; ++round) {
        state.waveSizes.push_back(static_cast<uint32_t>(wave.size()));
        uint64_t edges_this_level = 0;
        std::vector<int32_t> upcoming;
        upcoming.reserve(wave.size() * 2);
        int32_t chosen = kDeviceInvalidCandidate;

        for (int32_t from : wave) {
            if (from < 0 || static_cast<uint32_t>(from) >= packed.vertexCount) continue;
            const int32_t begin = packed.rowBegin[static_cast<size_t>(from)];
            const int32_t end = packed.rowBegin[static_cast<size_t>(from) + 1];
            edges_this_level += static_cast<uint64_t>(std::max(0, end - begin));
            for (int32_t cursor = begin; cursor < end; ++cursor) {
                const int32_t to = packed.neighbor[static_cast<size_t>(cursor)];
                if (to < 0 || static_cast<uint32_t>(to) >= packed.vertexCount) continue;
                if (seenVertices[static_cast<size_t>(to)] == 0) {
                    seenVertices[static_cast<size_t>(to)] = 1;
                    // 首次发现时记录下一层的层号。
                    state.level[static_cast<size_t>(to)] = static_cast<int32_t>(round + 1);
                    ++state.visited_count;
                    // 将新发现的顶点加入下一层。
                    upcoming.push_back(to);
                    if (packed.available[static_cast<size_t>(to)] == 1 && to < chosen) {
                        chosen = to;
                    }
                }
            }
        }

        state.processed_edges += edges_this_level;
        state.waveEdges.push_back(edges_this_level);
        if (chosen != kDeviceInvalidCandidate) {
            state.replacement = chosen;
            state.distance = static_cast<int32_t>(round + 1);
            break;
        }
        wave.swap(upcoming);
    }
    state.timing.total_us = watch.elapsed_us();
    return state;
}



### 4.5 NPU 辅助工程代码

下面的 Host、CMake 和运行脚本已经提供，不需要补全。

Host 调用任务一至三的代码准备 CSR，启动 NPU 扩展当前层的邻居，再调用第 4.3 节的 `buildNextFrontier` 过滤已访问顶点并构造下一层。每次正式运行均与第 4.4 节的 CPU 参考结果核对候选、距离和全部层号；出现不一致时输出 FAIL，并返回非零值。


In [ ]:
%%writefile Source/02.06/ascend_ops/host_launch/practice_npu_main.cpp
#include <acl/acl.h>
#include <aclrtlaunch_neighbor_expand.h>

#include "practice_search.hpp"
#include "practice_frontier.hpp"

#include <cstdint>
#include <cstdlib>
#include <dlfcn.h>
#include <iomanip>
#include <iostream>
#include <stdexcept>
#include <string>
#include <vector>

#define ACL_CHECK(expr)                                                              \
    do {                                                                             \
        const aclError _ret = (expr);                                                 \
        if (_ret != ACL_SUCCESS) {                                                    \
            std::string _message = std::string("ACL error: ") + #expr +                \
                                   ", code=" + std::to_string(static_cast<int>(_ret)); \
            const char* _detail = aclGetRecentErrMsg();                                \
            if (_detail && *_detail) _message += std::string("\n") + _detail;           \
            if (_ret == ACL_ERROR_RT_DEV_SETUP_ERROR) {                               \
                _message += "\nDevice startup failed. Check npu-smi info, the "       \
                            "assigned device mapping, and the CANN/driver runtime."; \
            }                                                                        \
            throw std::runtime_error(_message);                                       \
        }                                                                            \
    } while (0)

namespace {

void showDeviceEnvironment() {
    Dl_info library{};
    if (dladdr(reinterpret_cast<const void*>(&aclInit), &library) != 0 && library.dli_fname) {
        std::cout << "[INFO] AscendCL library=" << library.dli_fname << '\n';
        if (std::string(library.dli_fname).find("/stub/") != std::string::npos) {
            throw std::runtime_error("A link-time stub library was loaded. Remove stub paths "
                                     "from LD_LIBRARY_PATH and rebuild against the CANN runtime.");
        }
    }
    for (const char* name : {"ASCEND_RT_VISIBLE_DEVICES", "ASCEND_VISIBLE_DEVICES"}) {
        const char* value = std::getenv(name);
        std::cout << "[INFO] " << name << '=' << (value ? value : "<unset>") << '\n';
    }
    std::cout << std::flush;
}

void verifyDeviceIndex(int32_t device) {
    uint32_t count = 0;
    ACL_CHECK(aclrtGetDeviceCount(&count));
    std::cout << "[INFO] ACL visible device count=" << count
              << ", selected device index=" << device << std::endl;
    if (count == 0) {
        throw std::runtime_error("No NPU device is visible to this process. Check the "
                                 "Notebook resource allocation and container device mapping.");
    }
    if (device < 0 || static_cast<uint32_t>(device) >= count) {
        throw std::runtime_error("--device must be a visible device index in [0, " +
                                 std::to_string(count - 1) + "]. Check the assigned device "
                                 "mapping before selecting an index.");
    }
}

struct RuntimeOptions : public ExerciseConfig {
    int32_t device = 0;
    bool check_device_only = false;
};

void usage(const char* argv0) {
    std::cout << "Usage: " << argv0 << " [options]\n"
              << "Options:\n"
              << "  --device <id>                visible device index, default: 0\n"
              << "  --check-device               check device startup and exit\n"
              << "  --nodes <num>                node count, default: 262144\n"
              << "  --avg-degree <num>           target average undirected degree, default: 32\n"
              << "  --fault-node <id>            source faulty node, default: 0\n"
              << "  --health-percent <0..100>    probability of healthy nodes, default: 1\n"
              << "  --fault-radius <num>         force nodes within this BFS radius unavailable, default: 4\n"
              << "  --frontier-block-len <num>   static Tensor frontier block length, default: 32\n"
              << "  --neighbor-tile-len <num>    static Tensor neighbor tile length, default: 32\n"
              << "  --block-dim <num>            AI Core launch blockDim, default: 16\n"
              << "  --warmup <num>               warmup count, default: 2\n"
              << "  --repeat <num>               repeat count, default: 10\n"
              << "  --seed <num>                 random seed, default: 1234\n"
              << "  --max-depth <num>            BFS depth limit, default: nodes\n"
              << "  --csv <path>                 result CSV\n"
              << "  --sweep                      test frontier block len 256/512/1024/2048\n"
              << "  --print-graph                print a small CSR sample\n"
              << "  --print-levels               print first 32 output levels\n"
              << "  -h, --help                   show help\n";
}
RuntimeOptions readSettings(int argc, char** argv) {
    RuntimeOptions options;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto need_value = [&](const std::string& name) -> const char* {
            if (i + 1 >= argc) throw std::invalid_argument("missing value after " + name);
            return argv[++i];
        };
        if (arg == "--device") {
            options.device = std::stoi(need_value(arg));
        } else if (arg == "--check-device") {
            options.check_device_only = true;
        } else if (arg == "--nodes") {
            options.vertexCount = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--avg-degree") {
            options.degreeTarget = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--fault-node") {
            options.origin = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--health-percent") {
            options.healthyRate = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--fault-radius") {
            options.blockedHops = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--frontier-block-len") {
            options.frontierChunk = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--neighbor-tile-len") {
            options.neighborChunk = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--block-dim") {
            options.tasks = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--warmup") {
            options.preheat = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--repeat") {
            options.trials = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--seed") {
            options.randomSeed = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--max-depth") {
            options.depthBound = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--csv") {
            options.csvFile = need_value(arg);
        } else if (arg == "--sweep") {
            options.sweep = true;
        } else if (arg == "--print-graph") {
            options.showGraph = true;
        } else if (arg == "--print-levels") {
            options.showLevel = true;
        } else if (arg == "-h" || arg == "--help") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    return options;
}

std::vector<int32_t> make_padded_colidx(const PracticeCsr& packed, uint32_t neighborTileLen) {
    const uint32_t padded = alignElements(static_cast<uint32_t>(packed.neighbor.size()) + neighborTileLen, neighborTileLen);
    std::vector<int32_t> out(padded, -1);
    std::copy(packed.neighbor.begin(), packed.neighbor.end(), out.begin());
    return out;
}

std::vector<int32_t> make_padded_frontier(const std::vector<int32_t>& wave, uint32_t capacity, uint32_t blockLen) {
    const uint32_t active = static_cast<uint32_t>(wave.size());
    const uint32_t padded = std::max<uint32_t>(blockLen, alignElements(std::max(active, 1u), blockLen));
    if (padded > capacity) {
        throw std::runtime_error("frontier capacity exceeded");
    }
    std::vector<int32_t> out(padded, -1);
    std::copy(wave.begin(), wave.end(), out.begin());
    return out;
}

uint32_t count_visited(const std::vector<int32_t>& level) {
    uint32_t vertexTotal = 0;
    for (int32_t to : level) vertexTotal += (to >= 0 ? 1u : 0u);
    return vertexTotal;
}

struct DeviceBuffers {
    void* rowPtr = nullptr;
    void* colIdx = nullptr;
    void* wave = nullptr;
    void* nextBitmap = nullptr;
};

void free_buffers(DeviceBuffers& b) {
    if (b.nextBitmap) aclrtFree(b.nextBitmap);
    if (b.wave) aclrtFree(b.wave);
    if (b.colIdx) aclrtFree(b.colIdx);
    if (b.rowPtr) aclrtFree(b.rowPtr);
    b = DeviceBuffers{};
}
SearchState performDeviceSearch(const RuntimeOptions& options,
                                    const PracticeCsr& graph,
                                    const std::vector<int32_t>& colIdxPadded,
                                    DeviceBuffers& dev,
                                    aclrtStream stream,
                                    bool timed) {
    const uint32_t vertexTotal = graph.vertexCount;
    const uint32_t numEdgesPadded = static_cast<uint32_t>(colIdxPadded.size());
    const uint32_t frontierCapacity = alignElements(vertexTotal + options.frontierChunk, options.frontierChunk);
    const size_t frontierBytes = static_cast<size_t>(frontierCapacity) * sizeof(int32_t);
    const uint32_t wordsPerBitmap = (((vertexTotal + 31u) >> 5) + 15u) / 16u * 16u;
    const size_t nextBitmapBytes = static_cast<size_t>(options.tasks) * wordsPerBitmap * sizeof(uint32_t);

    std::vector<int32_t> visitedHost(vertexTotal, 0);
    std::vector<int32_t> levelHost(vertexTotal, kUnvisited);
    visitedHost[options.origin] = 1;
    levelHost[options.origin] = 0;

    // Host keeps visited/level as authoritative state. The Device expands
    // the current frontier and writes per-core compact bitsets. This keeps the
    // race-free layer semantics of per-core discovery while reducing transfer
    // volume by 32x compared with int32 per-core bitmaps.

    SearchState state;
    std::vector<int32_t> wave{static_cast<int32_t>(options.origin)};
    std::vector<uint32_t> nextBitmap(static_cast<size_t>(options.tasks) * wordsPerBitmap, 0);
    std::vector<uint32_t> mergedBitmap(wordsPerBitmap, 0);

    Stopwatch totalTimer;
    double kernelUs = 0.0;
    double frontierUs = 0.0;
    const uint32_t depthLimit = depthCeiling(options);

    for (uint32_t round = 0; !wave.empty() && round < depthLimit; ++round) {
        state.waveSizes.push_back(static_cast<uint32_t>(wave.size()));

        Stopwatch frontierTimer;
        auto paddedFrontier = make_padded_frontier(wave, frontierCapacity, options.frontierChunk);
        const uint32_t frontierPaddedCount = static_cast<uint32_t>(paddedFrontier.size());
        const size_t paddedFrontierBytes = static_cast<size_t>(frontierPaddedCount) * sizeof(int32_t);
        ACL_CHECK(aclrtMemcpy(dev.wave, frontierBytes, paddedFrontier.data(), paddedFrontierBytes, ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemset(dev.nextBitmap, nextBitmapBytes, 0, nextBitmapBytes));
        if (timed) frontierUs += frontierTimer.elapsed_us();

        Stopwatch kernelTimer;
        ACLRT_LAUNCH_KERNEL(neighbor_expand)(options.tasks, stream,
                                              dev.rowPtr, dev.colIdx, dev.wave, dev.nextBitmap,
                                              vertexTotal, numEdgesPadded,
                                              static_cast<uint32_t>(wave.size()), frontierPaddedCount,
                                              options.frontierChunk, options.neighborChunk,
                                              options.tasks);
        ACL_CHECK(aclrtSynchronizeStream(stream));
        if (timed) kernelUs += kernelTimer.elapsed_us();

        Stopwatch postTimer;

        uint64_t edgesThisLevel = 0;
        for (int32_t from : wave) {
            if (from < 0 || static_cast<uint32_t>(from) >= vertexTotal) continue;
            const int32_t begin = graph.rowBegin[static_cast<size_t>(from)];
            const int32_t end = graph.rowBegin[static_cast<size_t>(from) + 1];
            if (end > begin) edgesThisLevel += static_cast<uint64_t>(end - begin);
        }
        state.waveEdges.push_back(edgesThisLevel);
        state.processed_edges += edgesThisLevel;

        ACL_CHECK(aclrtMemcpy(nextBitmap.data(), nextBitmapBytes, dev.nextBitmap,
                              nextBitmapBytes, ACL_MEMCPY_DEVICE_TO_HOST));

        std::fill(mergedBitmap.begin(), mergedBitmap.end(), 0u);
        for (uint32_t c = 0; c < options.tasks; ++c) {
            const size_t base = static_cast<size_t>(c) * wordsPerBitmap;
            for (uint32_t w = 0; w < wordsPerBitmap; ++w) {
                mergedBitmap[w] |= nextBitmap[base + w];
            }
        }

        std::vector<int32_t> upcoming;
        upcoming.reserve(std::min<size_t>(vertexTotal, std::max<size_t>(wave.size() * 2, 64)));
        const int32_t best = buildNextFrontier(graph, options.origin, round,
                                              mergedBitmap, visitedHost, levelHost,
                                              upcoming);

        if (best != kDeviceInvalidCandidate) {
            state.replacement = best;
            state.distance = static_cast<int32_t>(round + 1);
            if (timed) frontierUs += postTimer.elapsed_us();
            break;
        }

        wave.swap(upcoming);
        if (timed) frontierUs += postTimer.elapsed_us();
    }

    state.timing.total_us = timed ? totalTimer.elapsed_us() : 0.0;
    state.timing.kernel_us = kernelUs;
    state.timing.host_frontier_us = frontierUs;
    state.level = levelHost;
    state.visited_count = count_visited(state.level);

    return state;
}

bool assessDeviceSearch(const RuntimeOptions& options, const PracticeCsr& graph, const SearchState& ref, bool append=false) {
    verifySettings(options);

    const uint32_t vertexTotal = graph.vertexCount;
    const auto colIdxPadded = make_padded_colidx(graph, options.neighborChunk);
    const uint32_t frontierCapacity = alignElements(vertexTotal + options.frontierChunk, options.frontierChunk);

    DeviceBuffers dev;
    aclrtStream stream = nullptr;

    const size_t rowPtrBytes = static_cast<size_t>(vertexTotal + 1) * sizeof(int32_t);
    const size_t colIdxBytes = static_cast<size_t>(colIdxPadded.size()) * sizeof(int32_t);
    const size_t frontierBytes = static_cast<size_t>(frontierCapacity) * sizeof(int32_t);
    const uint32_t wordsPerBitmap = (((vertexTotal + 31u) >> 5) + 15u) / 16u * 16u;
    const size_t nextBitmapBytes = static_cast<size_t>(options.tasks) * wordsPerBitmap * sizeof(uint32_t);

    try {
    Stopwatch setupTimer;
    ACL_CHECK(aclrtCreateStream(&stream));
    ACL_CHECK(aclrtMalloc(&dev.rowPtr, rowPtrBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.colIdx, colIdxBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.wave, frontierBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.nextBitmap, nextBitmapBytes, ACL_MEM_MALLOC_HUGE_FIRST));

    ACL_CHECK(aclrtMemcpy(dev.rowPtr, rowPtrBytes, graph.rowBegin.data(), rowPtrBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(dev.colIdx, colIdxBytes, colIdxPadded.data(), colIdxBytes, ACL_MEMCPY_HOST_TO_DEVICE));

    std::cout << "static_graph_setup_us=" << setupTimer.elapsed_us() << "\n";
    for (uint32_t i = 0; i < options.preheat; ++i) {
        (void)performDeviceSearch(options, graph, colIdxPadded, dev, stream, false);
    }

    SearchState last;
    SearchTiming avg;
    bool allPass = true;
    const uint32_t trials = std::max(1u, options.trials);
    for (uint32_t i = 0; i < trials; ++i) {
        last = performDeviceSearch(options, graph, colIdxPadded, dev, stream, true);
        const auto c = checkSearch(last, ref);
        allPass = allPass && c.replacement_ok && c.distance_ok && c.level_ok;
        avg.total_us += last.timing.total_us;
        avg.kernel_us += last.timing.kernel_us;
        avg.host_frontier_us += last.timing.host_frontier_us;
    }
    avg.total_us /= trials;
    avg.kernel_us /= trials;
    avg.host_frontier_us /= trials;
    last.timing = avg;
    const auto check = checkSearch(last, ref);
    const std::string status = allPass ? "PASS" : "FAIL";
    showTimingRow(graph, options.origin, last, status, true);
    const std::string method = options.sweep ? "NPU_BFS_L" + std::to_string(options.frontierChunk) : "NPU_BFS";
    saveTimingCsv(options.csvFile, method, avg.total_us, status, append);
    std::cout << "diagnosis: replacement=" << last.replacement
              << ", distance=" << last.distance << '\n';
    if (!allPass) std::cerr << "BFS validation failed; last level mismatches=" << check.level_mismatch << '\n';
    if (options.showLevel) showLayers(last);

    free_buffers(dev);
    aclrtDestroyStream(stream);
    return allPass;
    } catch (...) {
        if (stream) (void)aclrtSynchronizeStream(stream);
        free_buffers(dev);
        if (stream) (void)aclrtDestroyStream(stream);
        throw;
    }
}

}  // namespace

int main(int argc, char** argv) {
    bool initialized=false, deviceSet=false, argumentsValid=false;
    int device=0;
    try {
        const RuntimeOptions options = readSettings(argc, argv);
        verifySettings(options);
        argumentsValid=true;

        // 在构图前检查设备；设备初始化不计入 BFS 执行时间。
        showDeviceEnvironment();
        ACL_CHECK(aclInit(nullptr)); initialized=true;
        device=options.device;
        verifyDeviceIndex(device);
        ACL_CHECK(aclrtSetDevice(device)); deviceSet=true;
        if (options.check_device_only) {
            ACL_CHECK(aclrtResetDevice(device)); deviceSet=false;
            ACL_CHECK(aclFinalize()); initialized=false;
            std::cout << "Device initialization PASSED\n";
            return 0;
        }

        Stopwatch prepTimer;
        const auto graph = assembleNetwork(options);
        const double prepUs = prepTimer.elapsed_us();
        for (uint32_t i = 0; i < options.preheat; ++i)
            (void)searchSerial(graph, options.origin, depthCeiling(options));
        SearchState ref;
        double serialTotal = 0.0;
        for (uint32_t i = 0; i < options.trials; ++i) {
            ref = searchSerial(graph, options.origin, depthCeiling(options));
            serialTotal += ref.timing.total_us;
        }
        ref.timing.total_us = serialTotal / options.trials;

        
        describeNetwork(graph, options.origin);
        std::cout << "reference: replacement=" << ref.replacement
                  << ", distance=" << ref.distance
                  << ", visited=" << ref.visited_count
                  << ", processed_edges=" << ref.processed_edges
                  << ", preprocess_us=" << std::fixed << std::setprecision(2) << prepUs
                  << ", serial_bfs_us=" << ref.timing.total_us << std::defaultfloat << "\n";

        showTimingHeader(false);
        showTimingRow(graph, options.origin, ref, "REFERENCE", false);
        saveTimingCsv("results/bfs_cpu.csv", "CPU_BFS", ref.timing.total_us, "REFERENCE");

        showTimingHeader(true);
        bool allPass = true;
        bool append = false;
        if (options.sweep) {
            for (uint32_t len : {256u, 512u, 1024u, 2048u}) {
                RuntimeOptions cur = options;
                cur.frontierChunk = len;
                allPass = assessDeviceSearch(cur, graph, ref, append) && allPass;
                append = true;
            }
        } else {
            allPass = assessDeviceSearch(options, graph, ref);
        }

        ACL_CHECK(aclrtResetDevice(device)); deviceSet=false;
        ACL_CHECK(aclFinalize()); initialized=false;
        return allPass ? 0 : 2;
    } catch (const std::exception& e) {
        if (deviceSet) (void)aclrtResetDevice(device);
        if (initialized) (void)aclFinalize();
        std::cerr << "error: " << e.what() << "\n";
        if (!argumentsValid) usage(argv[0]);
        return 1;
    }
}


In [ ]:
%%writefile Source/02.06/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(bfs_practice LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

set(RUN_MODE "npu" CACHE STRING "Ascend C run mode")
set(SOC_VERSION "$ENV{SOC_VERSION}" CACHE STRING "Ascend SOC version")
set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
if(NOT ASCEND_CANN_PATH OR NOT SOC_VERSION)
  message(FATAL_ERROR "Use scripts/run_npu.sh to detect CANN and SOC_VERSION")
endif()
set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
else()
  message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}")
endif()

include("${ASCENDC_CMAKE_FILE}")

ascendc_library(practice_bfs_kernel STATIC
    ascend_ops/op_kernel/neighbor_expand.cpp
)
ascendc_compile_definitions(practice_bfs_kernel PRIVATE -DASCENDC_DUMP=0)


add_executable(bfs_practice_npu
    ascend_ops/host_launch/practice_npu_main.cpp
    cpp/practice_graph.cpp
    cpp/practice_build.cpp
    cpp/practice_search.cpp
)

target_include_directories(bfs_practice_npu PRIVATE
    ${CMAKE_CURRENT_SOURCE_DIR}/cpp
    ${ASCEND_CANN_PACKAGE_PATH}/include
    ${ASCEND_CANN_PACKAGE_PATH}/include/external
    ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
    ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp
    ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/tikcfw
    ${CMAKE_INSTALL_PREFIX}/include/practice_bfs_kernel
    ${CMAKE_BINARY_DIR}/out/include/practice_bfs_kernel
)

target_link_directories(bfs_practice_npu PRIVATE
    ${ASCEND_CANN_PACKAGE_PATH}/lib64
    ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
    ${ASCEND_CANN_PACKAGE_PATH}/acllib/lib64
    ${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/devlib
    ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/devlib
)

target_compile_options(bfs_practice_npu PRIVATE
    $<$<CXX_COMPILER_ID:GNU,Clang>:-O2 -Wall -Wextra -Wno-deprecated-declarations>
)

target_link_libraries(bfs_practice_npu PRIVATE
    practice_bfs_kernel
    ascendcl
    platform
    ascendalog
    c_sec
    dl
)

add_dependencies(bfs_practice_npu
    practice_bfs_kernel
)


In [ ]:
%%writefile Source/02.06/scripts/run_npu.sh
#!/usr/bin/env bash
set -euo pipefail
ROOT=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${ROOT}/build_ascend"
CANN_HINT=""
SOC_HINT="${SOC_VERSION:-}"
DEVICE_ID=0
while getopts ":a:v:d:h" opt; do
  case "${opt}" in
    a) CANN_HINT="${OPTARG}" ;;
    v) SOC_HINT="${OPTARG}" ;;
    d) DEVICE_ID="${OPTARG}" ;;
    h) echo "Usage: $0 [-a CANN_PATH] [-v SOC] [-d DEVICE] [-- program options]"; exit 0 ;;
    *) echo "Invalid option; use -h" >&2; exit 1 ;;
  esac
done
shift $((OPTIND - 1))
# CANNLab 环境检测与兼容配置。
is_cann_install_path() {
  local path="$1"
  [[ -f "${path}/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
     -f "${path}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
     -f "${path}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ]]
}

detect_cann_install_path() {
  local arch base candidate
  arch=$(uname -m)
  # 指定安装目录时只使用该目录，避免切换版本时回退到其他 CANN。
  if [[ -n "${1:-}" ]]; then
    for candidate in "${1}/${arch}-linux" "${1}"; do
      if is_cann_install_path "${candidate}"; then
        printf '%s\n' "${candidate}"
        return 0
      fi
    done
    return 1
  fi
  for base in "${ASCEND_INSTALL_PATH:-}" \
              "${ASCEND_HOME_PATH:-}" "${ASCEND_TOOLKIT_HOME:-}" \
              /opt/conda/Ascend/cann-* /usr/local/Ascend/ascend-toolkit/latest \
              /usr/local/Ascend/ascend-toolkit/*; do
    [[ -n "${base}" ]] || continue
    for candidate in "${base}/${arch}-linux" "${base}"; do
      if is_cann_install_path "${candidate}"; then
        printf '%s\n' "${candidate}"
        return 0
      fi
    done
  done
  return 1
}

detect_soc_version() {
  local raw_name normalized
  command -v npu-smi >/dev/null 2>&1 || return 1
  raw_name=$(npu-smi info 2>/dev/null \
    | sed -nE 's/^\|[[:space:]]*[0-9]+[[:space:]]+([^|]+)\|.*/\1/p' \
    | head -n 1 || true)
  normalized=$(printf '%s' "${raw_name}" | tr '[:upper:]' '[:lower:]' | tr -cd '[:alnum:]')
  case "${normalized}" in
    ascend910*|ascend310*) printf '%s\n' "${normalized}" ;;
    910*|310*) printf 'ascend%s\n' "${normalized}" ;;
    *) return 1 ;;
  esac
}

prepare_cann_environment() {
  local BUILD_DIR="$1" path_hint="${2:-}" soc_hint="${3:-${SOC_VERSION:-}}"
  local detected env_file="" candidate
  if [[ -n "${path_hint}" && ! -d "${path_hint}" ]]; then
    echo "CANN installation does not exist: ${path_hint}" >&2
    return 1
  fi
  if ! detected=$(detect_cann_install_path "${path_hint}"); then
    echo "Cannot find CANN Ascend C package. Set ASCEND_INSTALL_PATH or use -a <path>." >&2
    return 1
  fi
  ASCEND_INSTALL_PATH="${detected}"
  for candidate in "${ASCEND_INSTALL_PATH}/set_env.sh" "${ASCEND_INSTALL_PATH}/../set_env.sh"; do
    if [[ -f "${candidate}" ]]; then env_file="${candidate}"; break; fi
  done
  if [[ -n "${env_file}" ]]; then
    # Vendor environment scripts may read unset variables.
    set +u
    source "${env_file}"
    set -u
  fi
  # Keep the architecture-specific package selected before sourcing.
  ASCEND_INSTALL_PATH="${detected}"
  if [[ -n "${soc_hint}" ]]; then
    SOC_VERSION="${soc_hint}"
  elif detected_soc=$(detect_soc_version); then
    SOC_VERSION="${detected_soc}"
  else
    echo "Cannot detect SOC_VERSION with npu-smi. Set SOC_VERSION or use -v <soc>." >&2
    return 1
  fi
  mkdir -p "${BUILD_DIR}"
ASCEND_CMAKE_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
ASCENDC_CMAKE_NEW_LAYOUT="${ASCEND_INSTALL_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
BISHENG_REAL="${ASCEND_INSTALL_PATH}/ccec_compiler/bin/bisheng"
BISHENG_EXPECTED="${ASCEND_INSTALL_PATH}/ascendc_devkit/ccec_compiler/bin/bisheng"

if [[ -f "${ASCENDC_CMAKE_NEW_LAYOUT}" && -x "${BISHENG_REAL}" && ! -x "${BISHENG_EXPECTED}" ]]; then
  ASCEND_CMAKE_PACKAGE_PATH="${BUILD_DIR}/cann_package_compat"
  mkdir -p "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit"

  for entry in "${ASCEND_INSTALL_PATH}"/*; do
    [[ -e "${entry}" ]] || continue
    name=$(basename "${entry}")
    [[ "${name}" == "ascendc_devkit" ]] && continue
    ln -sfn "${entry}" "${ASCEND_CMAKE_PACKAGE_PATH}/${name}"
  done

  if [[ -d "${ASCEND_INSTALL_PATH}/ascendc_devkit" ]]; then
    for entry in "${ASCEND_INSTALL_PATH}/ascendc_devkit"/*; do
      [[ -e "${entry}" ]] || continue
      name=$(basename "${entry}")
      [[ "${name}" == "ccec_compiler" ]] && continue
      ln -sfn "${entry}" "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit/${name}"
    done
  fi

  for component in asc tikcpp ccec_compiler; do
    if [[ -e "${ASCEND_INSTALL_PATH}/${component}" ]]; then
      ln -sfn "${ASCEND_INSTALL_PATH}/${component}" \
        "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit/${component}"
    fi
  done
  echo "[INFO] Using CANN package compatibility view: ${ASCEND_CMAKE_PACKAGE_PATH}"
fi

export ASCEND_INSTALL_PATH
  export ASCEND_CANN_PACKAGE_PATH="${ASCEND_CMAKE_PACKAGE_PATH}"
  export SOC_VERSION
  export LD_LIBRARY_PATH="${ASCEND_INSTALL_PATH}/lib64:${ASCEND_INSTALL_PATH}/runtime/lib64:${LD_LIBRARY_PATH:-}"
  echo "[INFO] ASCEND_INSTALL_PATH=${ASCEND_INSTALL_PATH}"
  echo "[INFO] SOC_VERSION=${SOC_VERSION}"
}

prepare_cann_environment "${BUILD_DIR}" "${CANN_HINT}" "${SOC_HINT}"
cmake -S "${ROOT}" -B "${BUILD_DIR}" -DCMAKE_BUILD_TYPE=Release \
  -DRUN_MODE=npu -DSOC_VERSION="${SOC_VERSION}" \
  -DASCEND_CANN_PATH="${ASCEND_CANN_PACKAGE_PATH}" \
  -DASCEND_CANN_PACKAGE_PATH="${ASCEND_CANN_PACKAGE_PATH}"
cmake --build "${BUILD_DIR}" -j
cd "${ROOT}"
mkdir -p results
"${BUILD_DIR}/bfs_practice_npu" --device "${DEVICE_ID}" --csv results/bfs_npu.csv "$@"


In [ ]:
(PRACTICE_DIR / "scripts/run_npu.sh").chmod(0o755)
subprocess.run(["bash", "-n", str(PRACTICE_DIR / "scripts/run_npu.sh")], check=True)
print("NPU 工程文件已生成，运行脚本语法检查通过。")


### 4.6 编译并运行

完成前面任务后，执行下方脚本会检测 CANN 环境并构建全部 Host 与 Device 代码，在选定设备上执行相同规模的网络诊断。只有每次正式运行均通过校验，才显示 `Task 4 PASSED`。

**注：**
- Host 首先检查当前进程可见的设备数量及所选编号，并尝试初始化设备。设备编号按当前进程可见的索引填写；容器环境中的索引可能与 `npu-smi` 显示的物理编号不同。若出现 `507033`，表示设备启动失败，程序会同时输出底层错误信息，此时尚未进入图构建和 NPU BFS。应结合 `npu-smi info`、设备分配和 CANN/驱动日志排查运行环境。

- 需要单独检查设备初始化时，可运行 `bash Source/02.06/scripts/run_npu.sh -d 0 -- --check-device`。该命令不执行 BFS，也不生成性能结果。


In [ ]:
import subprocess
for path in sorted((PRACTICE_DIR / "scripts").glob("*.sh")):
    path.chmod(0o755)
    subprocess.run(["bash", "-n", str(path)], check=True)
# 填写当前进程可见的设备索引；容器映射后可能与物理编号不同。
PRACTICE_DEVICE_ID = 0
for name in ("bfs_cpu.csv", "bfs_npu.csv"):
    (PRACTICE_DIR / "results" / name).unlink(missing_ok=True)
log_path = PRACTICE_DIR / "results/task4.log"
with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(["bash", str(PRACTICE_DIR / "scripts/run_npu.sh"),
                             "-d", str(PRACTICE_DEVICE_ID)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
    returncode = proc.wait()
if returncode:
    raise RuntimeError(f"Task 4 构建或运行失败，返回码 {returncode}；请检查上方日志。")

import csv
from IPython.display import display, Markdown
table_rows = []
for kind in ['cpu', 'npu']:
    path = PRACTICE_DIR / 'results' / f'bfs_{kind}.csv'
    with path.open(newline='', encoding='utf-8') as stream:
        table_rows.extend(csv.DictReader(stream))
assert all(row['status'] in ['REFERENCE', 'PASS'] for row in table_rows)
table = ['| method | avg_us | status |', '|---|---:|---|']
table += [f"| {r['method']} | {float(r['avg_us']):.3f} | {r['status']} |" for r in table_rows]
display(Markdown('\n'.join(table)))
values = {r['method']: float(r['avg_us']) for r in table_rows}
if values.get('CPU_BFS', 0) > 0 and values.get('NPU_BFS', 0) > 0:
    display(Markdown(f"CPU/NPU 时间比：**{values['CPU_BFS']/values['NPU_BFS']:.3f}**。"))
print('Task 4 PASSED')


---
## 5. 实践总结

完成四项任务后，应能够说明同一张图在不同阶段如何表示和处理：

| 实践任务 | 核心内容 |
|---|---|
| 定义邻接表 | 定义边结点与顶点的指针成员，理解链式连接关系 |
| 图的构建 | 将新边结点插入表头，建立无向边的双向邻接关系 |
| 图的 CSR 压缩存储 | 沿链表累计边界，把邻接位置写入连续数组 |
| 基于 Ascend C 实现并行 BFS | 同层顶点分给多个任务，各自标记后由 Host 合并，并与 CPU 串行参考比较 |
